# 🕵️ Dia 85 — Recebimento do caso e planejamento

**Investigador Cyber 90 Dias** · Fase 7 · Rank Jutsu `S`

> Baseado no *Day 85* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-7-capstone-career/day-85.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).
> **Material sintético.** O *case packet* (Orrin Valley / Castellan) e a Operação Isca são ficção; domínios `.example`, IPs de documentação.

---

### 🎬 Onde paramos

A Marta mandou cinco perguntas às 7h40. Você respondeu a primeira com "foi a Lia, claro", e Rafa devolveu: *"Qual evidência **elimina** as outras três explicações?"*

Neste notebook você vai:

1. **Montar a matriz ACH em Python**, com a mesma regra da coluna F e o mesmo `I count` da planilha.
2. **Preencher células** de quatro linhas do capstone e ver o diagnóstico na hora.
3. **Rodar a análise de sensibilidade** (passo 8 do fluxograma) automaticamente.
4. **Carregar a ACH da Operação Isca** e encontrar o empate.
5. **Exportar** a matriz em CSV (abre no Calc) e a página de recebimento em Markdown.

🐍 Só biblioteca padrão do Python. Funciona offline: se a rede falhar, o notebook usa a cópia embutida dos arquivos. Rode as células de cima para baixo (Shift+Enter).

## 1. A regra da coluna F, em Python

A fórmula do original:

`=IF(OR(COUNTIF(B2:E2;"I")=0;COUNTIF(B2:E2;"I")=COUNTA(B2:E2));"non-diagnostic";"diagnostic")`

Traduzindo: uma linha **sem nenhum I** ou **com I em todas as hipóteses preenchidas** não separa nada. Repare que `COUNTA` conta células **preenchidas**, então um `N` conta como preenchido.

In [ ]:
def diagnostica(linha):
    """linha: dict {hipótese: 'C'|'I'|'N'}. Mesma lógica da coluna F do Calc."""
    n_i = sum(1 for v in linha.values() if v == "I")      # COUNTIF(...;"I")
    n_preenchidas = sum(1 for v in linha.values() if v)   # COUNTA(...)
    return not (n_i == 0 or n_i == n_preenchidas)         # nenhum I, ou I em tudo -> não diagnóstica

def contagem_i(matriz, hipoteses):
    """A linha 'I count' da planilha: quantos I cada hipótese recebeu."""
    return {h: sum(1 for l in matriz.values() if l["cel"][h] == "I") for h in hipoteses}

def lider(matriz, hipoteses):
    """Regra de Heuer: menos evidência CONTRA (menos I). Devolve lista, porque pode empatar."""
    cont = contagem_i(matriz, hipoteses)
    menor = min(cont.values())
    return [h for h in hipoteses if cont[h] == menor]

# Teste com as duas linhas genéricas do segundo diagrama do original
print("Ea (C,C,C):", "diagnóstica" if diagnostica({"HA": "C", "HB": "C", "HC": "C"}) else "não diagnóstica")
print("Eb (C,I,C):", "diagnóstica" if diagnostica({"HA": "C", "HB": "I", "HC": "C"}) else "não diagnóstica")

## 2. 🎮 A matriz do capstone (pergunta 3: de onde veio o M1?)

Hipóteses do original:

- **H1**: do lado da Castellan.
- **H2**: da caixa do Jordan, por acesso anterior a 9/mar que a auditoria não cobre.
- **H3**: da caixa compartilhada `ap@` ou de outro destinatário da Orrin Valley.
- **H4**: de outro jeito (um terceiro que trata o e-mail de uma das empresas).

A linha **E1** vem resolvida do original. As outras são evidências que o próprio original manda olhar (datas de registro, a carta, os destinatários, o log). Para cada célula, o notebook pergunta **C, I ou N**. Se você digitar algo inválido (ou rodar com `MODO_JOGO = False`), ele usa uma **leitura sugerida**, que é só um ponto de partida: discorde e escreva o seu motivo.

In [ ]:
HIPOTESES = ["H1", "H2", "H3", "H4"]

# Cada evidência: texto, fonte e uma leitura SUGERIDA com motivo por célula.
# A sugestão não é gabarito: é o raciocínio que você deve conferir (ou derrubar).
MATRIZ = {
    "E1": {"txt": "In-Reply-To do phishing = Message-ID exato de M1", "fonte": "P2; P3 M1",
           "cel": {"H1": "C", "H2": "C", "H3": "C", "H4": "C"},
           "motivo": "Quem tem uma cópia de M1 tem o Message-ID. Não diz de quem era a cópia."},
    "E2": {"txt": "castellan-hardwood.example registrado em 24/fev, ANTES de M1 existir (3/mar)", "fonte": "P6 L1; P7",
           "cel": None, "sugestao": {"H1": "C", "H2": "C", "H3": "C", "H4": "C"},
           "motivo": "Mostra preparo prévio contra a Castellan, mas qualquer origem do M1 cabe depois."},
    "E3": {"txt": "Carta falsa criada em 26/fev, endereçada 'To our valued customers'", "fonte": "P3 M3 metadados",
           "cel": None, "sugestao": {"H1": "C", "H2": "I", "H3": "I", "H4": "C"},
           "motivo": "Um modelo genérico para os CLIENTES da Castellan, feito antes da fatura, combina mal com um ataque que começou na caixa de um cliente."},
    "E4": {"txt": "O phishing foi só para o Jordan; M1 foi para ap@ com o Jordan em cópia", "fonte": "P2; P3 M1; P7",
           "cel": None, "sugestao": {"H1": "C", "H2": "C", "H3": "C", "H4": "C"},
           "motivo": "Qualquer cópia de M1 mostra o Jordan como destinatário. Não separa."},
    "E5": {"txt": "P8 (desde 9/mar) não mostra acesso do atacante ao jpike antes de 10/mar 14:31", "fonte": "P8",
           "cel": None, "sugestao": {"H1": "C", "H2": "C", "H3": "C", "H4": "C"},
           "motivo": "H2 fala de acesso ANTES de 9/mar, que o log não cobre. A ausência só pesa no período coberto."},
    "E6": {"txt": "Castellan (ouvir dizer): 'outros clientes' perguntaram de mudança bancária", "fonte": "P1 Parte C",
           "cel": None, "sugestao": {"H1": "C", "H2": "I", "H3": "I", "H4": "C"},
           "motivo": "Outros clientes da Castellan visados apontam para o lado da Castellan ou para um terceiro comum. Fonte fraca (hearsay)."},
}

MODO_JOGO = True
for eid, ev in MATRIZ.items():
    if ev["cel"] is not None:              # E1 já vem resolvida
        continue
    print(f"\n{eid}: {ev['txt']}  [{ev['fonte']}]")
    cel = {}
    for h in HIPOTESES:
        r = input(f"   {h}: C, I ou N? ").strip().upper()[:1] if MODO_JOGO else ""
        cel[h] = r if r in ("C", "I", "N") else ev["sugestao"][h]    # inválido -> sugestão
    ev["cel"] = cel
    print(f"   sua linha: {cel}   | sugestão: {ev['sugestao']}")
    print(f"   motivo sugerido: {ev['motivo']}")

In [ ]:
# A matriz como a planilha mostraria, com a coluna F e a linha 'I count'
print(f"{'Evid':5} " + " ".join(f"{h:3}" for h in HIPOTESES) + "  Diagnóstica?")
for eid, ev in MATRIZ.items():
    marcas = " ".join(f"{ev['cel'][h]:3}" for h in HIPOTESES)
    print(f"{eid:5} {marcas}  {'diagnostic' if diagnostica(ev['cel']) else 'non-diagnostic'}")
cont = contagem_i(MATRIZ, HIPOTESES)
print(f"{'I cnt':5} " + " ".join(f"{cont[h]:<3}" for h in HIPOTESES))
print("\nLiderança provisória (menos I):", lider(MATRIZ, HIPOTESES))

## 3. Sensibilidade: qual linha mudaria a liderança?

O passo 8 do fluxograma. Removemos uma linha de cada vez e recalculamos. A linha que muda o líder é a que merece uma segunda fonte, e é candidata à seção de riscos do relatório no Dia 89.

In [ ]:
def sensibilidade(matriz, hipoteses):
    base = lider(matriz, hipoteses)
    sensiveis = []
    for eid in matriz:
        sem = {k: v for k, v in matriz.items() if k != eid}   # a matriz sem esta linha
        novo = lider(sem, hipoteses)
        if novo != base:
            sensiveis.append((eid, novo))
    return base, sensiveis

base, sens = sensibilidade(MATRIZ, HIPOTESES)
print("Liderança com todas as linhas:", base)
if not sens:
    print("Nenhuma linha sozinha muda a liderança.")
for eid, novo in sens:
    print(f"⚠️  {eid} é sensível: sem ela, a liderança vira {novo}. Fonte: {MATRIZ[eid]['fonte']}")

> 🧠 Repare se a sua liderança depende de uma linha de **ouvir dizer** (E6). Se depende, a sua conclusão tem a força da fonte mais fraca. É exatamente isso que o Dia 89 vai pedir para você calibrar.

## 4. ➕ A ACH da Operação Isca

Pergunta: *"como o operador obteve os celulares dos cooperados?"* Os dados vêm de `caso-continuo/dados/dia-85/ach-isca.csv`.

In [ ]:
# Dados sintéticos da Operação Isca deste dia (plano B embutido).
ISCA_EMBUTIDO = {
"ach-isca.csv": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\n# H1 = base de cooperados vazou de um fornecedor da cooperativa (ex.: empresa de disparo de SMS)\n# H2 = o operador comprou uma base genérica vazada (telefones de MG) e disparou para todos\n# H3 = disparo aleatório em faixas de números, sem base nenhuma\n# H4 = o operador usou o chatbot Lia para validar celulares e CPFs antes do disparo\nid,evidencia,H1,H2,H3,H4,fonte,motivo\nE1,O texto do SMS cita 'Serra Azul' pelo nome,C,C,C,C,I2,qualquer hipotese explica citar a marca\nE2,\"Das 37 reclamacoes, 37 vieram de cooperados (nenhuma de nao cooperado)\",C,I,I,C,ouvidoria,base generica ou disparo aleatorio atingiriam muitos nao cooperados e parte deles ligaria para a cooperativa\nE3,Registros da Lia mostram perguntas que confirmam se um celular e de cooperado antes dos disparos,N,C,I,C,arco da Fase 6,disparo aleatorio nao precisaria validar nada\nE4,Os disparos duraram 1h28 (07h12 a 08h40) e vieram de 4 remetentes,C,C,C,C,I2,volume e janela cabem em todas as hipoteses\nE5,O fornecedor de SMS da cooperativa nao relatou incidente,C,C,C,C,relato da Marta (nao verificado),ausencia de relato nao e ausencia de incidente; e hearsay\nE6,Cooperados que nunca deram celular a cooperativa nao receberam SMS (amostra de 12 ligacoes da ouvidoria),C,I,I,C,ouvidoria,base generica e disparo aleatorio teriam atingido esses numeros\n"
}

In [ ]:
import urllib.request

RAW_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-85/"     # pasta caso-continuo/dados/dia-85 no GitHub

ISCA = {}
for nome, copia in ISCA_EMBUTIDO.items():
    try:                                                     # plano A: versão publicada no repo
        with urllib.request.urlopen(RAW_ISCA + nome, timeout=10) as r:
            ISCA[nome] = r.read().decode("utf-8")
        print(f"{nome}: baixado do GitHub")
    except Exception:                                        # plano B: cópia embutida
        ISCA[nome] = copia
        print(f"{nome}: cópia embutida")

In [ ]:
import csv, io

linhas = [l for l in ISCA["ach-isca.csv"].splitlines() if not l.startswith("#")]   # tira comentários
ISCA_H = ["H1", "H2", "H3", "H4"]
ACH_ISCA = {}
for row in csv.DictReader(io.StringIO("\n".join(linhas))):
    ACH_ISCA[row["id"]] = {"txt": row["evidencia"], "fonte": row["fonte"],
                           "cel": {h: row[h] for h in ISCA_H}, "motivo": row["motivo"]}

for eid, ev in ACH_ISCA.items():
    marcas = " ".join(ev["cel"][h] for h in ISCA_H)
    print(f"{eid}  {marcas}  {'diag.    ' if diagnostica(ev['cel']) else 'NÃO diag.'}  {ev['txt'][:70]}")
print("\nI count:", contagem_i(ACH_ISCA, ISCA_H))
b, s = sensibilidade(ACH_ISCA, ISCA_H)
print("Liderança:", b, "<- empate!" if len(b) > 1 else "")

**Empate entre H1 (fornecedor) e H4 (Lia).** Nenhuma linha atual separa os dois. A evidência que desempata é **temporal**: se a validação de números na Lia aconteceu *antes* de qualquer lista existir com o operador, H4 ganha força; se o operador já tinha a lista e só *conferiu* alguns números, H1 e H4 podem ser verdadeiras juntas. Isso vira pedido de dados à Marta: logs do fornecedor de SMS e a janela exata das consultas na Lia.

## 5. 📝 Exportar a matriz e a página de recebimento

In [ ]:
#@title Recebimento { display-mode: "form" }
pergunta_marta_1 = "Como o operador obteve os celulares? -> Que fonte de números explica quem recebeu o SMS? Resposta esperada: parcial"  #@param {type:"string"}
pergunta_marta_3 = "Foi alguém de dentro? -> Há evidência de ação que facilitou o golpe nas fontes entregues? Resposta esperada: parcial, sem nomes"  #@param {type:"string"}
pergunta_marta_4 = "Quanto dá para recuperar? -> Nenhuma pela Vigia; MED via instituição e polícia"  #@param {type:"string"}
pedido_dados = "1. Logs do fornecedor de SMS (envios de 01 a 05/out). 2. Janela exata das consultas suspeitas na Lia."  #@param {type:"string"}

from pathlib import Path
Path("notas").mkdir(exist_ok=True)

def exportar_csv(matriz, hips, caminho):
    """CSV com as mesmas colunas da planilha do original (abre direto no Calc)."""
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["Evidence"] + hips + ["Diagnostic?", "Source", "Reason"])
        for eid, ev in matriz.items():
            w.writerow([f"{eid}: {ev['txt']}"] + [ev["cel"][h] for h in hips] +
                       ["diagnostic" if diagnostica(ev["cel"]) else "non-diagnostic", ev["fonte"], ev["motivo"]])
        cont = contagem_i(matriz, hips)
        w.writerow(["I count"] + [cont[h] for h in hips])

exportar_csv(MATRIZ, HIPOTESES, "notas/ach-q3.csv")
exportar_csv(ACH_ISCA, ISCA_H, "notas/ach-isca.csv")
intake = (f"# Recebimento — Operação Isca\n\n## Perguntas\n- {pergunta_marta_1}\n- {pergunta_marta_3}\n- {pergunta_marta_4}\n\n"
          f"## Pedido de dados\n{pedido_dados}\n\n## ACH\nLiderança: {b}. Ver notas/ach-isca.csv.\n")
Path("notas/intake-isca.md").write_text(intake, encoding="utf-8")
print(intake)
try:
    from google.colab import files
    for a in ("notas/ach-q3.csv", "notas/ach-isca.csv", "notas/intake-isca.md"):
        files.download(a)
except ImportError:
    print("(Fora do Colab: arquivos em notas/)")

## ✅ Antes de fechar o Dia 85

- [ ] Cinco perguntas do cliente reescritas (capstone) e cinco da Marta (Isca), com resposta esperada e motivo
- [ ] Matriz do capstone com 6+ linhas e pelo menos uma não diagnóstica
- [ ] Linha sensível identificada e transformada em pedido de dados
- [ ] Empate da ACH do Isca explicado, com o pedido que desempata
- [ ] Tentações nomeadas, cada uma com a regra que impede

**Amanhã:** Dia 86: forense. Três logs, três relógios, uma linha do tempo UTC. A âncora decide o fuso, não o palpite da TI.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>